# Simple image training using CLIP model

This notebook fine-tunes a [CLIP](https://huggingface.co/sentence-transformers/clip-ViT-B-32) model on [`sentence-transformers/unsplash-lite`](https://huggingface.co/datasets/sentence-transformers/unsplash-lite): ~25k Unsplash photos, each paired with descriptive keywords. We treat every `(image, keywords)` pair as a positive and train with `MultipleNegativesRankingLoss`, which uses the other captions in each batch as negatives. This is the standard contrastive objective for CLIP-style models. We hold out 100 pairs to track text-to-image retrieval quality during training.

In [ ]:
from datasets import load_dataset

from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments

In [2]:
# Load CLIP model
model = SentenceTransformer("sentence-transformers/clip-ViT-B-32")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 4701.37it/s]


In [3]:
# Load the Unsplash Lite dataset: ~25k photos, each paired with descriptive keywords
dataset = load_dataset("sentence-transformers/unsplash-lite", split="train")
dataset

Dataset({
    features: ['image', 'keywords'],
    num_rows: 24996
})

In [4]:
# Join the ";"-separated keywords into a comma-separated caption to use as the text side of each pair
def keywords_to_caption(batch):
    return {"caption": [keywords.replace(";", ", ") for keywords in batch["keywords"]]}


dataset = dataset.map(keywords_to_caption, batched=True, remove_columns=["keywords"])
# MultipleNegativesRankingLoss reads columns in order, so (image, caption) becomes (anchor, positive)
dataset = dataset.select_columns(["image", "caption"])

# Hold out 100 pairs for evaluation
dataset = dataset.train_test_split(test_size=100, seed=42)
train_dataset, eval_dataset = dataset["train"], dataset["test"]
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'caption'],
        num_rows: 24896
    })
    test: Dataset({
        features: ['image', 'caption'],
        num_rows: 100
    })
})

In [ ]:
# Preview a training pair
sample = train_dataset[0]
print(sample["caption"])
sample["image"]

In [6]:
# Evaluate text-to-image retrieval on the held-out pairs: each caption should retrieve its own image
queries = {idx: sample["caption"] for idx, sample in enumerate(eval_dataset)}
corpus = {idx: sample["image"] for idx, sample in enumerate(eval_dataset)}
relevant_docs = {idx: [idx] for idx in range(len(eval_dataset))}
dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="unsplash-dev",
)

# Baseline metrics for the un-finetuned model
dev_evaluator(model)

{'unsplash-dev_cosine_accuracy@1': 0.56,
 'unsplash-dev_cosine_accuracy@3': 0.74,
 'unsplash-dev_cosine_accuracy@5': 0.81,
 'unsplash-dev_cosine_accuracy@10': 0.88,
 'unsplash-dev_cosine_precision@1': 0.56,
 'unsplash-dev_cosine_precision@3': 0.24666666666666665,
 'unsplash-dev_cosine_precision@5': 0.162,
 'unsplash-dev_cosine_precision@10': 0.08799999999999997,
 'unsplash-dev_cosine_recall@1': 0.56,
 'unsplash-dev_cosine_recall@3': 0.74,
 'unsplash-dev_cosine_recall@5': 0.81,
 'unsplash-dev_cosine_recall@10': 0.88,
 'unsplash-dev_cosine_ndcg@10': 0.7142141400917714,
 'unsplash-dev_cosine_mrr@10': 0.661607142857143,
 'unsplash-dev_cosine_map@100': 0.6679418915747591}

In [7]:
# Each image is matched against its own caption (positive) and the other captions in the batch (in-batch negatives)
train_loss = MultipleNegativesRankingLoss(model)

In [8]:
# Fine-tune CLIP on the image/caption pairs, evaluating retrieval every 10% of training
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="output/clip-unsplash-lite",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=False,  # Set to True if your GPU does not support BF16
    bf16=True,  # Set to False if your GPU does not support BF16
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy="no",
    logging_steps=0.05,
    logging_first_step=True,
    run_name="clip-unsplash-lite",  # Used in W&B if `wandb` is installed
)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=train_loss,
    evaluator=dev_evaluator,
)
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 49407, 'bos_token_id': 49406, 'pad_token_id': 49407}.


Step,Training Loss,Validation Loss,Unsplash-dev Cosine Accuracy@1,Unsplash-dev Cosine Accuracy@3,Unsplash-dev Cosine Accuracy@5,Unsplash-dev Cosine Accuracy@10,Unsplash-dev Cosine Precision@1,Unsplash-dev Cosine Precision@3,Unsplash-dev Cosine Precision@5,Unsplash-dev Cosine Precision@10,Unsplash-dev Cosine Recall@1,Unsplash-dev Cosine Recall@3,Unsplash-dev Cosine Recall@5,Unsplash-dev Cosine Recall@10,Unsplash-dev Cosine Ndcg@10,Unsplash-dev Cosine Mrr@10,Unsplash-dev Cosine Map@100
78,0.793050,0.827728,0.620000,0.810000,0.880000,0.910000,0.620000,0.270000,0.176000,0.091000,0.620000,0.810000,0.880000,0.910000,0.771835,0.726706,0.730510
156,0.785378,0.816179,0.600000,0.830000,0.860000,0.920000,0.600000,0.276667,0.172000,0.092000,0.600000,0.830000,0.860000,0.920000,0.763553,0.712929,0.716025
234,0.743295,0.726830,0.640000,0.800000,0.870000,0.930000,0.640000,0.266667,0.174000,0.093000,0.640000,0.800000,0.870000,0.930000,0.778377,0.730361,0.732585
312,0.707115,0.606615,0.700000,0.840000,0.890000,0.930000,0.700000,0.280000,0.178000,0.093000,0.700000,0.840000,0.890000,0.930000,0.812012,0.774123,0.776561
390,0.670243,0.666043,0.690000,0.830000,0.920000,0.920000,0.690000,0.276667,0.184000,0.092000,0.690000,0.830000,0.920000,0.920000,0.812281,0.776667,0.779160
468,0.628005,0.660642,0.680000,0.800000,0.890000,0.910000,0.680000,0.266667,0.178000,0.091000,0.680000,0.800000,0.890000,0.910000,0.793766,0.756083,0.759436
546,0.595550,0.597104,0.690000,0.890000,0.910000,0.940000,0.690000,0.296667,0.182000,0.094000,0.690000,0.890000,0.910000,0.940000,0.828074,0.790774,0.792482
624,0.537299,0.607798,0.720000,0.860000,0.900000,0.920000,0.720000,0.286667,0.180000,0.092000,0.720000,0.860000,0.900000,0.920000,0.824796,0.793595,0.796093
702,0.506453,0.570526,0.680000,0.870000,0.910000,0.920000,0.680000,0.290000,0.182000,0.092000,0.680000,0.870000,0.910000,0.920000,0.814319,0.778833,0.781757
778,0.486290,0.547862,0.720000,0.870000,0.900000,0.930000,0.720000,0.290000,0.180000,0.093000,0.720000,0.870000,0.900000,0.930000,0.833538,0.801706,0.803688


TrainOutput(global_step=778, training_loss=0.6669286613599194, metrics={'train_runtime': 253.5942, 'train_samples_per_second': 98.173, 'train_steps_per_second': 3.068, 'total_flos': 0.0, 'train_loss': 0.6669286613599194, 'epoch': 1.0})

In [ ]:
# Save the fine-tuned model
model.save_pretrained("output/clip-unsplash-lite/final")

# Optionally, share it on the Hugging Face Hub (requires `huggingface-cli login`)
# model.push_to_hub("clip-unsplash-lite")

In [ ]:
from IPython.display import clear_output, display

from sentence_transformers.util import semantic_search

# Re-embed the first 1,000 images with the fine-tuned model
images = train_dataset[:1000]["image"]
image_embeddings = model.encode(images, batch_size=32, convert_to_tensor=True, show_progress_bar=True)

# Retrieve the top 3 matching images for the typed keywords, until an empty line is entered
while True:
    query = input("Search images by keywords (leave empty to stop): ")
    if not query:
        break
    # Clear the previous results before showing the new ones
    clear_output(wait=True)
    query_embedding = model.encode(query, convert_to_tensor=True)
    hits = semantic_search(query_embedding, image_embeddings, top_k=3)[0]
    for hit in hits:
        print(f"Score: {hit['score']:.3f}")
        # Show each match at thumbnail size, preserving aspect ratio
        thumbnail = images[hit["corpus_id"]].copy()
        thumbnail.thumbnail((256, 256))
        display(thumbnail)